# Attention Deep-Dive — Hands-On

**LLM Engineering · Domain 2 · Roadmap Weeks 08/10**

Companion to `02 Literature Notes/LLM Engineering/Attention Deep-Dive`. Pure numpy, runs offline.
We build attention from scratch, verify the causal mask, and watch multi-head shapes.

## 0. Setup + softmax

In [ ]:
%pip install -q numpy
import numpy as np
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)
rng = np.random.RandomState(0)
print("ok")

## 1. Scaled dot-product attention + causal mask

In [ ]:
def attention(Q, K, V, causal=False):
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    if causal:
        m = np.triu(np.ones_like(scores), 1).astype(bool)
        scores = np.where(m, -1e9, scores)
    w = softmax(scores, -1)
    return w @ V, w

seq, d = 5, 8
Q = K = V = rng.randn(seq, d)
out, w = attention(Q, K, V, causal=True)
print("weights (lower-triangular, rows sum to 1):")
print(w.round(2))
assert np.allclose(w.sum(1), 1)
assert np.allclose(np.triu(w, 1), 0), "future leaked!"
print("causal mask verified: no token attends to the future")

## 2. Why we scale by √d_k
Watch softmax collapse toward one-hot as dimension grows if we DON'T scale.

In [ ]:
for dk in (4, 64, 1024):
    q = rng.randn(dk); k = rng.randn(10, dk)
    unscaled = softmax(k @ q)
    scaled   = softmax(k @ q / np.sqrt(dk))
    print(f"d_k={dk:>4}  max weight  unscaled={unscaled.max():.2f}  scaled={scaled.max():.2f}")

> Unscaled max weight rushes toward 1.0 (saturation) as d_k grows; scaling keeps it moderate.

## 3. Multi-head attention shapes

In [ ]:
def mha(X, Wq, Wk, Wv, Wo, h, causal=True):
    seq, d = X.shape; dk = d // h
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    split = lambda t: t.reshape(seq, h, dk).transpose(1, 0, 2)
    Q, K, V = split(Q), split(K), split(V)
    s = Q @ K.transpose(0, 2, 1) / np.sqrt(dk)
    if causal:
        m = np.triu(np.ones((seq, seq)), 1).astype(bool)
        s = np.where(m, -1e9, s)
    ctx = (softmax(s, -1) @ V).transpose(1, 0, 2).reshape(seq, d)
    return ctx @ Wo

seq, d, h = 6, 16, 4
X = rng.randn(seq, d)
Wq, Wk, Wv, Wo = (rng.randn(d, d) * 0.1 for _ in range(4))
print("input :", X.shape, " heads:", h, " d_k/head:", d // h)
print("output:", mha(X, Wq, Wk, Wv, Wo, h).shape, "(same as input width)")

## 4. The O(seq²) cost, measured

In [ ]:
for seq in (128, 256, 512, 1024):
    Q = rng.randn(seq, 64)
    ops = seq * seq * 64          # QKᵀ multiply-adds
    print(f"seq={seq:>4}  score-matrix cells={seq*seq:>9,}  ~QKᵀ flops={ops:>13,}")
print("doubling seq -> ~4x attention cost (quadratic)")

## 5. Exercises
1. Add per-head weight inspection — which head attends most locally vs globally?
2. Implement bidirectional (no mask) attention and compare weights.
3. Add positional encodings (sinusoidal) to X and see how weights shift.
4. Time attention for seq=2048 and confirm the ~4x scaling from seq=1024.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Attention Deep-Dive`
- Snippets: `04 Code Snippets/LLM/Scaled Dot-Product Attention in NumPy`, `.../Multi-Head Attention in NumPy`
- MOC: `06 Maps of Content/LLM Engineering Concepts`